# Deutsch–Jozsa

A promise function $f:\{0,1\}^n\to\{0,1\}$ is either **constant** (the same
bit on every input) or **balanced** (exactly half the inputs map to $0$).
Classically you may need $2^{n-1}+1$ evaluations. The quantum circuit uses
**one** oracle call.

Here $n=2$. After $H^{\otimes n}$ on the inputs, the phase-kickback oracle,
and another $H^{\otimes n}$, measuring the all-zero string means constant;
anything else means balanced.

Qiskit prints qubit 0 on the **right**. This notebook does not import the
sibling `.py` file.

In [ ]:
import qiskit as qk
import qiskit_aer as qka

N = 2
ANCILLA = N
print("classical worst case", 2 ** (N - 1) + 1, "queries")

## Oracles

$U_f$ implements $|x\rangle|y\rangle \mapsto |x\rangle|y\oplus f(x)\rangle$
on $n$ input qubits plus one ancilla.

- constant $0$: identity
- constant $1$: $X$ on the ancilla
- balanced $f(x)=x_0$: CNOT from qubit $0$ onto the ancilla
- balanced $f(x)=x_0\oplus x_1$: two CNOTs (parity)

In [ ]:
def oracle_constant_0() -> qk.QuantumCircuit:
    return qk.QuantumCircuit(N + 1, name="f=0")


def oracle_constant_1() -> qk.QuantumCircuit:
    qc = qk.QuantumCircuit(N + 1, name="f=1")
    qc.x(ANCILLA)
    return qc


def oracle_lsb() -> qk.QuantumCircuit:
    qc = qk.QuantumCircuit(N + 1, name="f=x0")
    qc.cx(0, ANCILLA)
    return qc


def oracle_parity() -> qk.QuantumCircuit:
    qc = qk.QuantumCircuit(N + 1, name="f=x0⊕x1")
    qc.cx(0, ANCILLA)
    qc.cx(1, ANCILLA)
    return qc


ORACLES = {
    "constant-0": ("constant", oracle_constant_0),
    "constant-1": ("constant", oracle_constant_1),
    "balanced-lsb": ("balanced", oracle_lsb),
    "balanced-parity": ("balanced", oracle_parity),
}

## Circuit

Prepare the ancilla in $|-\rangle$ ($X$ then $H$), Hadamard the inputs,
apply $U_f$, Hadamard the inputs again, measure only the $n$ inputs.

In [ ]:
def deutsch_jozsa(oracle: qk.QuantumCircuit) -> qk.QuantumCircuit:
    qc = qk.QuantumCircuit(N + 1, N)
    qc.x(ANCILLA)
    qc.h(range(N + 1))
    qc.compose(oracle, inplace=True)
    qc.h(range(N))
    qc.measure(range(N), range(N))
    return qc


def p_all_zero(oracle: qk.QuantumCircuit) -> float:
    qc = qk.QuantumCircuit(N + 1)
    qc.x(ANCILLA)
    qc.h(range(N + 1))
    qc.compose(oracle, inplace=True)
    qc.h(range(N))
    probs = qk.quantum_info.Statevector.from_instruction(qc).probabilities_dict()
    return sum(float(p) for bits, p in probs.items() if bits[-N:] == "0" * N)


qc = deutsch_jozsa(oracle_parity())
print(qc.draw())

## Verdict

$P(|00\rangle)\approx 1$ means constant. Near $0$ means balanced.

In [ ]:
sim = qka.AerSimulator()
for name, (promise, factory) in ORACLES.items():
    oracle = factory()
    p0 = p_all_zero(oracle)
    guess = "constant" if p0 > 0.5 else "balanced"
    counts = sim.run(qk.transpile(deutsch_jozsa(oracle), sim), shots=1024).result().get_counts()
    print(f"{name:18} promise={promise:8}  P(|00>)={p0:.3f}  guess={guess}  shots={counts}")

qk.visualization.plot_histogram(
    sim.run(qk.transpile(deutsch_jozsa(oracle_parity()), sim), shots=1024).result().get_counts()
)